# Stopword & noise-phrase list maintenance

Keeps the `public.stopwords` table up to date by mining
`public.ngrams_summary` for words and phrases (`n_gram` 1-4) that occur
almost every day across most sources — a strong signal of a
generic/function word or boilerplate phrase rather than a real trend.

The list is size-independent: single words and multi-word phrases live in
the same table and are matched against as one list in golang, regardless of
how many words an entry has.

Entries are never rewritten or removed - each run only *inserts* rows that
aren't already present (anything previously mined is treated as "known" on
the next run and won't be re-proposed). Query the `public.stopwords` table
to review or delete a run's additions.

**Setup (once):**
```bash
cd notebooks
uv sync
```


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ['POSTGRES_HOST'],
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    user=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'],
    dbname=os.environ['POSTGRES_DATABASE'],
    sslmode=os.getenv('PGSSLMODE', 'require'),
)


def q(sql: str, params: dict | None = None) -> pd.DataFrame:
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


STOPWORDS_TABLE = 'public.stopwords'
print('Ready')
print('Stopwords table:', STOPWORDS_TABLE)


## Parameters

- `languages`: which `ngrams_summary.language` values to mine.
- `window_days`: how far back to look for the ubiquity calculation.
- `min_day_ubiquity` / `min_total_freq`: thresholds for single words.
- `min_day_ubiquity_phrase` / `min_total_freq_phrase`: separate, stricter
  thresholds for multi-word phrases (`n_gram` 2-4) - phrase space is much
  bigger, so genuinely constant boilerplate is rarer and needs less evidence
  to trust than a single word does.
- `ALLOWLIST`: domain/topical words or phrases (any size) that must never be
  auto-added, even if they are statistically ubiquitous. Extend this over
  time.


In [ ]:
params = {
    'languages': ['en', 'de'],
    'window_days': 90,
    'min_day_ubiquity': 0.75,
    'min_total_freq': 300,
    'min_day_ubiquity_phrase': 0.6,
    'min_total_freq_phrase': 100,
}



## Mine `ngrams_summary` for ubiquitous single words

For each language, `day_ubiquity` is the fraction of days (within
`window_days`) a word was seen on at all, across any source. A word that
shows up almost every day, regardless of what's actually in the news, is a
function/generic word rather than a trend word.


In [ ]:
mining_sql = '''
WITH bounds AS (
    SELECT language, max(day) AS ref_day
    FROM public.ngrams_summary
    WHERE n_gram = 1
    GROUP BY language
),
recent AS (
    SELECT s.words, s.language, s.day, s.sources, s.frequencies
    FROM public.ngrams_summary s
    JOIN bounds b USING (language)
    WHERE s.n_gram = 1
      AND s.language = ANY(%(languages)s)
      AND s.day > b.ref_day - %(window_days)s
      AND s.words ~ '^[[:alpha:]]+$'
),
window_days AS (
    SELECT language, count(DISTINCT day) AS total_days
    FROM recent
    GROUP BY language
)
SELECT
    r.words,
    r.language,
    count(DISTINCT r.day)                                   AS days_seen,
    sum(r.sources)                                          AS source_hits,
    sum(r.frequencies)                                      AS total_freq,
    w.total_days,
    round(count(DISTINCT r.day)::numeric / w.total_days, 3) AS day_ubiquity
FROM recent r
JOIN window_days w USING (language)
GROUP BY r.words, r.language, w.total_days
HAVING sum(r.frequencies) >= %(min_total_freq)s
ORDER BY day_ubiquity DESC, total_freq DESC
'''

mined = q(mining_sql, params)
print(f"{len(mined)} single words pass the min_total_freq threshold")
mined.head(20)


## Filter down to genuinely new candidates

Apply `min_day_ubiquity`, drop anything already listed in the `stopwords`
table, and drop anything on `ALLOWLIST`. What's left is shown for review
below - these are the words that would be inserted into the table.


In [ ]:
def load_known() -> set[str]:
    return set(q(f'SELECT stopword FROM {STOPWORDS_TABLE}')['stopword'].str.lower())


known = load_known()

eligible = mined[mined['day_ubiquity'] >= params['min_day_ubiquity']]
candidates = (
    eligible[~eligible['words'].isin(known)]
    .sort_values(['day_ubiquity', 'total_freq'], ascending=False)
    .reset_index(drop=True)
)

print(f"{len(known)} entries already known, {len(eligible)} pass ubiquity threshold")
print(f"{len(candidates)} new candidate(s) - review before trusting blindly")
candidates


## Write back to the `stopwords` table

Insert-only: previously mined entries are never rewritten or removed - they
were already loaded into `known` above, so they're simply skipped this
time. Only genuinely new `candidates` are inserted, sorted.


In [ ]:
def append_new_entries(new_entries: list[str]) -> None:
    if not new_entries:
        print('No new candidates - nothing to insert.')
        return

    inserted = q(
        f'INSERT INTO {STOPWORDS_TABLE} (stopword, exact) '
        'SELECT unnest(%(words)s::text[]), TRUE '
        'RETURNING stopword',
        {'words': sorted(new_entries)},
    )

    print(f"Inserted {len(inserted)} new entrie(s) into {STOPWORDS_TABLE}")



append_new_entries(candidates['words'].tolist())

## Mine `ngrams_summary` for ubiquitous multi-word phrases

Same idea as the single-word mining above, but for `n_gram` 2-4: phrases
that appear on almost every day, across most sources, regardless of what's
actually in the news that day. Real trends are bursty by definition, so
near-constant phrases are boilerplate/navigation noise ("read more", "sign
up for", "cookie policy", ...), not trends - and get inserted into the same
`stopwords` table as the single words above.

Already-known noise (HTML/wiki/parser/style artifacts) is excluded up front
using the same regexes as [`ngrams.go`](../golang/internal/ngrams/ngrams.go)
and `trending.sql`, so this surfaces genuinely *new* noise phrases.


In [ ]:
mining_sql_phrases = '''
WITH bounds AS (
    SELECT language, max(day) AS ref_day
    FROM public.ngrams_summary
    WHERE n_gram BETWEEN 2 AND 4
    GROUP BY language
),
recent AS (
    SELECT s.words, s.n_gram, s.language, s.day, s.sources, s.frequencies
    FROM public.ngrams_summary s
    JOIN bounds b USING (language)
    WHERE s.n_gram BETWEEN 2 AND 4
      AND s.language = ANY(%(languages)s)
      AND s.day > b.ref_day - %(window_days)s
      AND s.words ~ '[[:alpha:]]'
      AND s.words !~ '^[0-9]+([[:space:]]+[0-9]+)*$'
      AND lower(s.words) !~ '(^|[[:space:]])(39|34|gt|lt)([[:space:]]|$)'
),
window_days AS (
    SELECT language, count(DISTINCT day) AS total_days
    FROM recent
    GROUP BY language
)
SELECT
    r.words,
    r.n_gram,
    r.language,
    count(DISTINCT r.day)                                   AS days_seen,
    sum(r.sources)                                          AS source_hits,
    sum(r.frequencies)                                      AS total_freq,
    w.total_days,
    round(count(DISTINCT r.day)::numeric / w.total_days, 3) AS day_ubiquity
FROM recent r
JOIN window_days w USING (language)
GROUP BY r.words, r.n_gram, r.language, w.total_days
HAVING sum(r.frequencies) >= %(min_total_freq_phrase)s
ORDER BY day_ubiquity DESC, total_freq DESC
'''

mined_phrases = q(mining_sql_phrases, params)
print(f"{len(mined_phrases)} phrases pass the min_total_freq_phrase threshold")
mined_phrases.head(20)

## Filter down to genuinely new phrase candidates

Apply `min_day_ubiquity_phrase`, drop anything already listed in the
`stopwords` table (reloaded to pick up words inserted above), and drop
anything on `ALLOWLIST`. What's left is shown for review below - these are
the phrases that would be inserted into the table.


In [ ]:
known = load_known()

eligible_phrases = mined_phrases[mined_phrases['day_ubiquity'] >= params['min_day_ubiquity_phrase']]
candidate_phrases = (
    eligible_phrases[
        ~eligible_phrases['words'].isin(known)
    ]
    .sort_values(['day_ubiquity', 'total_freq'], ascending=False)
    .reset_index(drop=True)
)

print(f"{len(known)} entries already known, {len(eligible_phrases)} pass ubiquity threshold")
print(f"{len(candidate_phrases)} new candidate(s) - review before trusting blindly")
candidate_phrases


## Review and write back to the `stopwords` table

Before writing, this step shows all `candidate_phrases` and lets you correct
what will be inserted:

- remove incorrect rows by index
- add corrected/new values manually
- confirm with `INSERT` to actually write

If you do not confirm, nothing is inserted.

In [ ]:
if candidate_phrases.empty:
    print("No phrase candidates to review or insert.")
else:
    review_df = (
        candidate_phrases[['words', 'day_ubiquity', 'total_freq']]
        .copy()
        .reset_index(drop=True)
    )
    review_df.index.name = 'row'

    print("Review phrase candidates before insert:")
    display(review_df)

    drop_raw = input(
        "Row indices to EXCLUDE (comma-separated, blank = keep all): "
    ).strip()
    add_raw = input(
        "Extra/corrected words to ADD (comma-separated, blank = none): "
    ).strip()

    words_to_insert = review_df['words'].tolist()

    if drop_raw:
        drop_idx = set()
        for token in drop_raw.split(','):
            token = token.strip()
            if not token:
                continue
            try:
                idx = int(token)
            except ValueError:
                print(f"Ignoring invalid row index: {token}")
                continue
            if idx < 0 or idx >= len(words_to_insert):
                print(f"Ignoring out-of-range row index: {idx}")
                continue
            drop_idx.add(idx)

        words_to_insert = [
            word for idx, word in enumerate(words_to_insert) if idx not in drop_idx
        ]

    if add_raw:
        extras = [w.strip().lower() for w in add_raw.split(',') if w.strip()]
        words_to_insert.extend(extras)

    # Deduplicate while preserving order.
    reviewed_words = []
    seen = set()
    for word in words_to_insert:
        if word not in seen:
            reviewed_words.append(word)
            seen.add(word)

    review_insert_df = pd.DataFrame({'words': reviewed_words})
    print(f"\nFinal reviewed insert set: {len(reviewed_words)} word(s)")
    display(review_insert_df)

    confirm = input("Type INSERT to write these words, or press Enter to cancel: ").strip()
    if confirm == 'INSERT':
        append_new_entries(reviewed_words)
    else:
        print("Insert cancelled. No rows were written.")